# Leccion 2: ARIMA - Nuestro primer modelo de prediccion

**Dataset**: GEFCom2014 - 3 anios de carga electrica horaria

**Objetivo**: Construir un modelo SARIMAX y predecir la carga electrica 3 horas adelante.

---

## Que es ARIMA?

ARIMA = **A**uto**R**egressive **I**ntegrated **M**oving **A**verage

Es un modelo que aprende de **3 fuentes**:

| Fuente | Que significa | Ejemplo |
|--------|---------------|--------|
| **AR** (AutoRegresivo) | Los valores PASADOS predicen el futuro | "La hora pasada fue 3000 MW, esta sera similar" |
| **I** (Integrated) | Diferencia los datos para quitar tendencia | "Resto el valor anterior para estabilizar" |
| **MA** (Moving Average) | Los ERRORES PASADOS corrigen la siguiente prediccion | "Me equivoque 200 MW, ahora corrijo" |

Como nuestra serie tiene **estacionalidad** (patron que se repite cada 24h), usamos **SARIMA** (Seasonal ARIMA) con parametros adicionales: `P, D, Q, m`.

## Analogia: Predecir el trafico

Imagina que quieres predecir el **trafico en una autopista**:

- **AR**: "A esta hora siempre hay trafico" (usa patrones pasados)
- **I**: "El trafico sube cada anio, necesito quitar esa tendencia"
- **MA**: "Ayer me equivoque, hoy corrijo"
- **SARIMA**: "Ademas, todos los lunes a las 8am hay trafico" (estacionalidad)

ARIMA combina estas 3 fuentes para hacer la mejor prediccion posible.

## 0. Instalar dependencias

In [ ]:
!pip install statsmodels -q

## 1. Imports - Que usamos y por que

| Libreria | Para que sirve |
|----------|----------------|
| `pandas` | Manejar datos en tablas |
| `matplotlib` | Hacer graficos |
| `numpy` | Operaciones numericas |
| `SARIMAX` | El modelo ARIMA estacional |
| `MinMaxScaler` | Escalar datos al rango [0,1] |
| `mape` | Medir error de prediccion |
| `load_data` | Cargar el dataset automaticamente |

In [ ]:
import os
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import datetime as dt
import math

from pandas.plotting import autocorrelation_plot
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import MinMaxScaler
from common.utils import load_data, mape
from IPython.display import Image

%matplotlib inline
pd.options.display.float_format = '{:,.2f}'.format
np.set_printoptions(precision=2)
warnings.filterwarnings('ignore')

print('Imports listos - SARIMAX, MinMaxScaler, mape')

## 2. Cargar los datos

Recordatorio de la Leccion 1:
- Tenemos **26,304 filas** (3 anios x 365 dias x 24 horas)
- Columna `load`: consumo electrico en Megawatts (MW)
- Frecuencia: **horaria** (cada fila = 1 hora)

In [ ]:
energy = load_data('./data')[['load']]
print(f'Filas: {energy.shape[0]}')
print(f'Columnas: {list(energy.columns)}')
print(f'Rango: {energy.index.min()} -> {energy.index.max()}')
energy.head(10)

In [ ]:
energy.plot(y='load', subplots=True, figsize=(15, 8), fontsize=12)
plt.xlabel('timestamp', fontsize=12)
plt.ylabel('load (MW)', fontsize=12)
plt.suptitle('Serie completa: Ene 2012 -> Dic 2014', fontsize=14)
plt.tight_layout()
plt.show()

### Que ves en el grafico?

1. **Estacionalidad anual**: picos en verano (jul-ago) y valles en primavera
2. **Estacionalidad diaria**: zig-zag cada 24 horas (dia/noche)
3. **Tendencia leve**: la serie se mantiene mas o menos estable

Esta estacionalidad es la razon por la que usamos **SARIMA** (con S de Seasonal) en vez de ARIMA simple.

## 3. Separar Train y Test (split TEMPORAL)

### Por que NO random split?

En problemas normales (clasificacion, regresion), hacemos `train_test_split` con `shuffle=True`.

En series temporales, **NUNCA** podemos hacer eso. Porque:

```
Si hago random split:
  Train: [2014-03-15, 2012-07-20, 2014-11-01, 2013-01-05, ...]  (mezclado)
  Test:  [2014-12-30, 2012-09-10, 2014-06-15, ...]  (mezclado)

El modelo "ve" el futuro y hace trampa!
```

### Como hacemos el split?

Usamos un **corte en el tiempo**:

```
Train: Nov 1 -> Dic 29, 2014  (1416 horas)
Test:  Dic 30 -> Dic 31, 2014 (48 horas = 2 dias)
```

Entrenamos con el pasado, predecimos el futuro. Asi como en la vida real.

In [ ]:
train_start_dt = '2014-11-01 00:00:00'
test_start_dt = '2014-12-30 00:00:00'

print(f'Train: {train_start_dt} -> {test_start_dt}')
print(f'Test:  {test_start_dt} -> 2014-12-31 23:00:00')

In [ ]:
# Visualizar train vs test
energy[(energy.index < test_start_dt) & (energy.index >= train_start_dt)][['load']]\
    .rename(columns={'load':'train'})\
    .join(energy[test_start_dt:][['load']].rename(columns={'load':'test'}), how='outer')\
    .plot(y=['train', 'test'], figsize=(15, 8), fontsize=12)

plt.xlabel('timestamp', fontsize=12)
plt.ylabel('load (MW)', fontsize=12)
plt.title('Train (naranja) vs Test (azul)', fontsize=14)
plt.legend(fontsize=12)
plt.show()

### Que ves?

- **Naranja** = Train (Nov 1 -> Dic 29): datos con los que entrenamos
- **Azul** = Test (Dic 30-31): los ultimos 2 dias que guardamos para evaluar

El modelo NUNCA ve los datos azules durante el entrenamiento. Los usamos solo para medir que tan bien predice.

## 4. Escalar los datos (MinMaxScaler)

### Por que escalar?

ARIMA funciona mejor cuando los datos estan en el rango **[0, 1]**. Es como poner todos los ingredientes en la misma escala antes de cocinar.

### Como funciona?

```
original = 3000 MW

escalado = (original - min) / (max - min)
         = (3000 - 1979) / (5224 - 1979)
         = 1021 / 3245
         = 0.315
```

### Regla de oro: fit en train, transform en test

- `fit_transform()` en train: aprende el rango y escala
- `transform()` en test: usa el mismo rango del train (no "ve" el futuro)

Si haces `fit_transform()` en test, el scaler "ve" los valores futuros y eso es trampa.

In [ ]:
train = energy.copy()[(energy.index >= train_start_dt) & (energy.index < test_start_dt)][['load']]
test = energy.copy()[energy.index >= test_start_dt][['load']]

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')

In [ ]:
# Escalar train
scaler = MinMaxScaler()
train['load'] = scaler.fit_transform(train)

# Verificar rango [0, 1]
print(f'Train min: {train["load"].min():.4f}')
print(f'Train max: {train["load"].max():.4f}')
train.head(10)

In [ ]:
# Comparar: original vs escalado
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

energy[(energy.index >= train_start_dt) & (energy.index < test_start_dt)][['load']]\
    .rename(columns={'load':'original'})\
    .plot.hist(bins=100, ax=axes[0], title='Original', fontsize=12)

train.rename(columns={'load':'escalado [0,1]'})\
    .plot.hist(bins=100, ax=axes[1], title='Escalado', fontsize=12)

plt.tight_layout()
plt.show()

### Que ves?

- **Izquierda**: Distribucion original (1979 a 5224 MW)
- **Derecha**: Distribucion escalada (0.0 a 1.0)

La forma es identica, solo cambio la escala. ARIMA trabaja mejor con datos en [0,1].

In [ ]:
# Escalar test (con el mismo scaler del train)
test['load'] = scaler.transform(test)
print(f'Test min: {test["load"].min():.4f}')
print(f'Test max: {test["load"].max():.4f}')
test.head()

## 5. Configurar SARIMAX

### Los 6 parametros clave

SARIMAX tiene **6 parametros** que debemos definir. Piensa en ellos como los "botones" del modelo:

| Parametro | Que controla | Ejemplo |
|-----------|-------------|---------|
| `p` | Cuantos valores PASADOS mira el modelo | 4 = mira las ultimas 4 horas |
| `d` | Cuantas veces DIFERENCIAR para quitar tendencia | 1 = resta el valor anterior una vez |
| `q` | Cuantos ERRORES PASADOS usa para corregir | 0 = no usa errores |
| `P` | Lags ESTACIONALES auto-regresivos | 1 = mira el valor de hace 24h |
| `D` | Diferenciacion ESTACIONAL | 1 = una vez |
| `Q` | Lags ESTACIONALES de media movil | 0 = ninguno |
| `m` | Periodo estacional | 24 = cada 24 horas |

### Nuestra configuracion

```
order = (4, 1, 0)           # p=4, d=1, q=0
seasonal_order = (1, 1, 0, 24)  # P=1, D=1, Q=0, m=24
```

### Por que estos valores?

- **p=4**: La hora pasada es la mejor prediccion de la hora futura
- **d=1**: La serie tiene leve tendencia, la quitamos con 1 diferenciacion
- **q=0**: No usamos correccion por errores (AR puro)
- **P=1**: Mira el valor de hace 24 horas (patron diario)
- **D=1**: La estacionalidad tambien tiene tendencia
- **Q=0**: No usamos errores estacionales
- **m=24**: El patron se repite cada 24 horas

In [ ]:
HORIZON = 3  # Predecir 3 horas adelante
print(f'Forecasting horizon: {HORIZON} horas')

order = (4, 1, 0)           # p=4, d=1, q=0
seasonal_order = (1, 1, 0, 24)  # P=1, D=1, Q=0, m=24

print(f'order: {order}')
print(f'seasonal_order: {seasonal_order}')

### Entender los parametros en detalle

**`order=(4, 1, 0)`** - Componente no-estacional:
- `p=4`: El modelo mira las ultimas 4 horas para predecir
  - Ejemplo: Para predecir hora 10, usa horas 6, 7, 8, 9
- `d=1`: Diferencia una vez para quitar tendencia
  - Ejemplo: En vez de usar 3000 MW, usa 3000 - 2950 = 50 MW
- `q=0`: No usa errores pasados (solo AR)

**`seasonal_order=(1, 1, 0, 24)`** - Componente estacional:
- `P=1`: Mira el valor de hace 24 horas (1 dia)
  - Ejemplo: Para predecir martes 10am, usa lunes 10am
- `D=1`: Diferencia estacional una vez
- `Q=0`: No usa errores estacionales
- `m=24`: El periodo es 24 horas (estacionalidad diaria)

In [ ]:
# Entrenar el modelo
model = SARIMAX(endog=train, order=order, seasonal_order=seasonal_order)
results = model.fit()

print(results.summary())

### Que es `results.summary()`?

Es el "reporte" del modelo. Contiene:
- **AIC/BIC**: Metricas de calidad del ajuste (menor = mejor)
- **Coeficientes**: Cuanto peso le da a cada lag
- **P-valores**: Si los coeficientes son estadisticamente significativos

Por ahora no te preocupes por todos los numeros. Lo importante es que el modelo se entreno sin errores.

## 6. Walk-Forward Validation

### Que es walk-forward?

Es el **gold standard** de validacion en series temporales. La idea es simple:

```
Paso 1: Entrena en [horas 1-720]  -> Predice hora 721
Paso 2: Entrena en [horas 2-721]  -> Predice hora 722
Paso 3: Entrena en [horas 3-722]  -> Predice hora 723
...
```

Cada vez que llega un dato nuevo, re-entrenamos el modelo. Asi como en produccion.

### Por que no simplemente predecir todo de una?

Porque en la vida real, cada hora llega un dato nuevo y queremos la mejor prediccion posible con la informacion mas reciente.

### Ventana de entrenamiento

Usamos una **ventana de 720 horas** (30 dias). Esto significa que el modelo solo mira el mes mas reciente, no toda la historia. Por que?

- Datos muy viejos pueden no ser relevantes
- Mas datos = mas tiempo de entrenamiento
- 30 dias es un buen balance

In [ ]:
# Preparar test data con columnas shifted para walk-forward
test_shifted = test.copy()

for t in range(1, HORIZON+1):
    test_shifted['load+'+str(t)] = test_shifted['load'].shift(-t, freq='H')

test_shifted = test_shifted.dropna(how='any')
print(f'Test shifted shape: {test_shifted.shape}')
test_shifted.head(5)

### Que es `test_shifted`?

Es el test data preparado para walk-forward. Cada fila tiene:

| Columna | Que significa |
|---------|---------------|
| `load` | Valor actual (target) |
| `load+1` | Valor 1 hora adelante |
| `load+2` | Valor 2 horas adelante |
| `load+3` | Valor 3 horas adelante |

Asi el loop puede comparar cada prediccion con el valor real.

In [ ]:
%%time
training_window = 720  # 30 dias (720 horas) de ventana de entrenamiento

train_ts = train['load']
test_ts = test_shifted

# Historial: ultimos 720 valores del train
history = [x for x in train_ts]
history = history[(-training_window):]

predictions = list()

# Walk-forward loop
for t in range(test_ts.shape[0]):
    # 1. Entrenar modelo con historial actual
    model = SARIMAX(endog=history, order=order, seasonal_order=seasonal_order)
    model_fit = model.fit()
    
    # 2. Predecir HORIZON pasos adelante
    yhat = model_fit.forecast(steps=HORIZON)
    predictions.append(yhat)
    
    # 3. Obtener valor real
    obs = list(test_ts.iloc[t])
    
    # 4. Mover ventana: agregar obs real, quitar el mas viejo
    history.append(obs[0])
    history.pop(0)
    
    # 5. Log cada paso
    print(f'{test_ts.index[t]} | paso {t+1}: predicho={yhat[:2]}... real={obs[:2]}...')

### Que acaba de pasar?

El loop hizo esto N veces (una por cada hora del test):

1. **Entreno** el modelo con las ultimas 720 horas
2. **Predijo** las proximas 3 horas
3. **Comparo** con el valor real
4. **Actualizo** el historial (agrego el real, quito el mas viejo)
5. **Repite**

Es como si cada hora, el modelo "aprendiera" de su error y mejorara.

## 7. Evaluar con MAPE

### Que es MAPE?

**MAPE** = Mean Absolute Percentage Error

```
MAPE = mean( |real - predicho| / real ) x 100%
```

### Como interpretarlo?

| MAPE | Que significa |
|------|---------------|
| 0.5% | Excelente: predije con 0.5% de error |
| 2% | Bueno: predije con 2% de error |
| 5% | Regular: predije con 5% de error |
| 10% | Malo: predije con 10% de error |

En series temporales, un MAPE < 2% es **muy bueno**.

In [ ]:
# Crear DataFrame de evaluacion
eval_df = pd.DataFrame(predictions, columns=['t+'+str(t) for t in range(1, HORIZON+1)])
eval_df['timestamp'] = test_ts.index[0:len(predictions)]
eval_df = pd.melt(eval_df, id_vars='timestamp', value_name='prediction', var_name='h')
eval_df['actual'] = np.array(np.transpose(test_ts)).ravel()

# Invertir escala: de [0,1] a MW originales
eval_df[['prediction', 'actual']] = scaler.inverse_transform(eval_df[['prediction', 'actual']])

eval_df.head()

In [ ]:
# MAPE por horizonte (t+1, t+2, t+3)
if HORIZON > 1:
    eval_df['APE'] = (eval_df['prediction'] - eval_df['actual']).abs() / eval_df['actual']
    print('MAPE por horizonte:')
    print(eval_df.groupby('h')['APE'].mean())
    print()

### Que ves?

El MAPE para **t+1** (1 hora adelante) es el mas bajo. Esto es logico:

- Predecir 1 hora adelante es mas facil
- Predecir 3 horas adelante es mas dificil

Mientras mas lejos predigas, mas error acumulas.

In [ ]:
# One-step MAPE (solo t+1)
one_step_mape = mape(
    eval_df[eval_df['h'] == 't+1']['prediction'],
    eval_df[eval_df['h'] == 't+1']['actual']
) * 100

print(f'One-step forecast MAPE: {one_step_mape:.2f}%')

In [ ]:
# Multi-step MAPE (todos los horizontes)
multi_step_mape = mape(eval_df['prediction'], eval_df['actual']) * 100
print(f'Multi-step forecast MAPE: {multi_step_mape:.2f}%')
print()
if multi_step_mape < 2:
    print('Excelente: menos de 2% de error')
elif multi_step_mape < 5:
    print('Bueno: menos de 5% de error')
else:
    print('Regular: mas de 5% de error - hay que mejorar')

## 8. Visualizar predicciones vs reales

El mejor para ver que tan bien predijo el modelo.

- **Rojo** = Valor real
- **Azul** = Prediccion (mas oscuro = mas confiable)

In [ ]:
if HORIZON == 1:
    eval_df.plot(x='timestamp', y=['actual', 'prediction'],
                 style=['r', 'b'], figsize=(15, 8))
else:
    # Preparar datos para grafico multi-step
    plot_df = eval_df[(eval_df.h=='t+1')][['timestamp', 'actual']]
    for t in range(1, HORIZON+1):
        plot_df['t+'+str(t)] = eval_df[(eval_df.h=='t+'+str(t))]['prediction'].values

    fig = plt.figure(figsize=(15, 8))
    ax = plt.plot(plot_df['timestamp'], plot_df['actual'],
                  color='red', linewidth=4.0, label='Actual')
    ax = fig.add_subplot(111)
    
    for t in range(1, HORIZON+1):
        x = plot_df['timestamp'][(t-1):]
        y = plot_df['t+'+str(t)][0:len(x)]
        ax.plot(x, y, color='blue',
                linewidth=4*math.pow(.9,t),
                alpha=math.pow(0.8,t),
                label=f'Prediccion t+{t}')

    ax.legend(loc='best')

plt.xlabel('timestamp', fontsize=12)
plt.ylabel('load (MW)', fontsize=12)
plt.title(f'Prediccion vs Real - MAPE: {multi_step_mape:.2f}%', fontsize=14)
plt.show()

### Que ves?

Las lineas azules siguen de cerca la roja. El modelo esta capturando:
- El patron diario (zig-zag cada 24h)
- Los picos y valles
- La tendencia general

Si las lineas azules estan lejos de la roja, el modelo necesita mejorar.

## 9. Resumen

### Flujo completo

```
Cargar datos -> Split temporal -> Escalar -> Configurar SARIMAX -> Walk-forward -> Evaluar
```

### Que aprendimos

| Concepto | Que es | Por que importa |
|----------|--------|------------------|
| **ARIMA** | AutoRegresivo + Integrado + Moving Average | Modelo base para series temporales |
| **SARIMAX** | ARIMA + Estacional + Features exogenas | Para series con patron repetitivo |
| **Walk-forward** | Re-entrenar con cada dato nuevo | Simular produccion real |
| **MAPE** | Error porcentual absoluto | Medir calidad de prediccion |
| **MinMaxScaler** | Escalar datos a [0,1] | ARIMA funciona mejor asi |
| **Split temporal** | Cortar por fecha, no random | No hacer trampa con el futuro |

### Proximo paso

Ahora que entendemos ARIMA, podemos compararlo con **SVR** (Support Vector Regressor) en la Leccion 3.

---

**Siguiente**: [Leccion 3: SVR](lesson-3-svr.md) - Modelo no-lineal alternativo